# 07 · Funktionen

**Lernziele**

- Du definierst eigene Funktionen mit Parametern und Rückgabewerten.
- Du weißt, wann eine Funktion `None` liefert, und nutzt `None` gezielt für „nichts gefunden“.
- Du nutzt Standardwerte, Schlüsselwort-Argumente und beliebig viele Argumente.
- Du dokumentierst Funktionen mit Docstrings und Type Hints.
- Du übergibst Funktionen als Werte und schreibst kleine Lambdas.
- Du löst Probleme rekursiv und erkennst, wann das sinnvoll ist.
- Du schreibst Generatoren mit `yield`.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2, 6, 8 und 16.

## Warum Funktionen?

Funktionen geben einem Stück Code einen **Namen**. Das macht Programme kürzer, lesbarer und testbar. Faustregel: Kopierst du Code ein zweites Mal, gehört er in eine Funktion.

## Funktionen definieren

In [ ]:
def greet(name):
    print(f"Hallo {name}!")


greet("Ada")
greet("Linus")

Mit `return` liefert eine Funktion ein Ergebnis zurück. Ohne `return` liefert sie `None`.

In [ ]:
def gross_price(net_price, vat_rate):
    return net_price * (1 + vat_rate)


price = gross_price(100, 0.19)
print(f"{price:.2f} €")

> `print` zeigt etwas an, `return` gibt etwas zurück. Nur mit `return` kannst du mit dem Ergebnis weiterrechnen.

## `None` als Rückgabewert

`None` steht für „kein Wert“; wie du darauf prüfst, steht in Notebook 03 (Abschnitt „None und Vergleiche“). Bei Funktionen begegnet dir `None` an drei Stellen.

Erstens liefert eine Funktion ohne `return` (oder mit leerem `return`) `None`. Typischer Fehler: Die Funktion druckt das Ergebnis, statt es zurückzugeben:

In [ ]:
def add_vat(net_price: float) -> None:
    print(f"{net_price * 1.19:.2f} €")  # druckt nur, gibt nichts zurück


total = add_vat(100)
print(total)  # None: mit total lässt sich nicht weiterrechnen

Zweitens liefern Methoden, die ein Objekt selbst verändern, `None`. `sort()` sortiert die Liste an Ort und Stelle; wer das Ergebnis wieder zuweist, verliert die Liste. `sorted()` liefert dagegen eine neue Liste:

In [ ]:
items = [3, 1, 2]
items = items.sort()
print(items)  # None: die Liste ist weg

items = [3, 1, 2]
print(sorted(items))  # neue, sortierte Liste
print(items)  # das Original bleibt unverändert

Drittens nutzt du `None` bewusst als Signal „nichts gefunden“: Die Funktion liefert den Treffer oder `None`, und der Type Hint `str | None` sagt das offen. Der Aufrufer prüft mit `is None`:

In [ ]:
def find_customer(customers: list[str], name: str) -> str | None:
    for customer in customers:
        if customer.lower() == name.lower():
            return customer
    return None  # nichts gefunden


customers = ["Ada", "Linus", "Grace"]
for name in ["grace", "Tim"]:
    customer = find_customer(customers, name)
    if customer is None:
        print(f"{name}: kein Kunde gefunden")
    else:
        print(f"{name}: gefunden als {customer}")

**Übung 1 (Rabattcode: in einem Satz erklären, leicht):** Lies `f` und beschreibe in **einem Satz**, was sie tut. Gib nur den Satz in die Chat-KI mit „Schreib eine Python-Funktion `f(total, code)`, die …“, kopiere die Antwort in die Zelle darunter und führe die Tests aus. Schlagen sie fehl, fehlte im Satz ein Randfall. Vergleiche danach: Wie geht die KI vor, wie das Original? Und was liefert das Original, wenn du `case _` weglässt?

Ohne KI-Zugang: Prüfe die typische KI-Antwort aus der Lösung.

In [ ]:
def f(total, code):
    match code:
        case "SPAR10":
            return round(total * 0.9, 2)
        case "MINUS5" if total >= 20:
            return total - 5
        case _:
            return total


assert f(42.0, "SPAR10") == 37.8
assert f(25.0, "MINUS5") == 20.0
assert f(15.0, "MINUS5") == 15.0  # Randfall: unter 20 € kein Abzug
assert f(30.0, "GRATIS") == 30.0  # Randfall: unbekannter Code ändert nichts

In [ ]:
# Typische KI-Antwort auf den Satz
# "Wendet einen Rabattcode an: SPAR10 gibt 10 % Rabatt, MINUS5 zieht 5 € ab.":
def f(total, code):
    if code == "SPAR10":
        return total * 0.9
    elif code == "MINUS5":
        return total - 5
    return total


print(f(42.0, "SPAR10") == 37.8)  # False: 37.800000000000004, ohne round
print(f(15.0, "MINUS5") == 15.0)  # False: 10.0, Mindestbestellwert fehlte im Satz
# Besserer Satz: "Gibt mit SPAR10 10 % Rabatt (auf Cent gerundet), zieht mit MINUS5
# ab 20 € Bestellwert 5 € ab und lässt den Betrag sonst unverändert."

# Vergleich: Die KI nimmt eine if-elif-Kette, das Original match mit Guard (if im case).
# Ohne case _ liefert das Original für "GRATIS" und für MINUS5 unter 20 € still None.

## Standardwerte und Schlüsselwort-Argumente

Parameter können einen **Standardwert** haben. Beim Aufruf darfst du Argumente auch **mit Namen** übergeben; dann ist die Reihenfolge egal.

In [ ]:
def gross_price(net_price, vat_rate=0.19):
    return net_price * (1 + vat_rate)


print(gross_price(100))
print(gross_price(100, 0.07))
print(gross_price(vat_rate=0.07, net_price=100))

> **Typischer Fehler:** Kein veränderbarer Standardwert wie `def f(items=[])`. Die Liste wird zwischen Aufrufen geteilt. Nimm stattdessen `None` und lege die Liste in der Funktion an.

## Mehrere Rückgabewerte

Eine Funktion kann ein Tupel zurückgeben, das du direkt entpackst:

In [ ]:
def min_max(numbers):
    return min(numbers), max(numbers)


lowest, highest = min_max([5, 2, 9, 1])
print(lowest, highest)

## Docstrings und Type Hints

Ein **Docstring** beschreibt, was die Funktion tut. **Type Hints** zeigen, welche Typen erwartet werden. Python prüft sie nicht, aber Editoren und Kollegen profitieren davon.

In [ ]:
def bmi(weight_kg: float, height_cm: float) -> float:
    """Berechnet den Body-Mass-Index aus Gewicht und Größe."""
    return weight_kg / (height_cm / 100) ** 2


help(bmi)

## Platzhalter mit `pass`

Ein Block darf nicht leer sein. `pass` ist eine Anweisung, die nichts tut, praktisch für Funktionen, die du später schreibst.

In [ ]:
def send_report():
    pass  # TODO: später umsetzen

## Gültigkeitsbereich

Variablen, die in einer Funktion entstehen, sind nur dort sichtbar.

In [ ]:
def calculate():
    local_result = 42
    return local_result


calculate()
print(local_result)

## Beliebig viele Argumente

Mit `*` vor einem Parameter nimmt eine Funktion beliebig viele Positionsargumente an. Sie landen in einem **Tupel**. Du kennst das von `print()`, das beliebig viele Werte ausgibt:

In [ ]:
def average(*numbers: float) -> float:
    return sum(numbers) / len(numbers)


print(average(1, 2, 3))
print(average(10, 20))

Mit `**` sammelst du beliebig viele **Schlüsselwort-Argumente** in einem Dictionary. Oft heißen die Parameter `*args` und `**kwargs`; die Namen sind nur Konvention, entscheidend sind `*` und `**`:

In [ ]:
def describe(name: str, **details: str) -> str:
    extras = ", ".join([f"{key}: {value}" for key, value in details.items()])
    return f"{name} ({extras})"


describe("Ada", born="1815", city="London")

Umgekehrt packt `*` beim **Aufruf** eine Liste in einzelne Argumente aus:

In [ ]:
values = [4, 8, 15]
print(average(*values))  # wie average(4, 8, 15)

**Übung 2 (Erinnerung an mehrere Empfänger, leicht):** Schreibe `send_reminder(message, *recipients, channel="E-Mail")`, die pro Empfänger eine Zeile wie `E-Mail an Ada: Teammeeting 10 Uhr` ausgibt. Rufe sie mit einzelnen Namen auf, dann mit der Liste `team` und `channel="Teams"`. Was passiert bei `send_reminder(message, team)` ohne Stern?

In [ ]:
team = ["Ada", "Grace", "Linus"]

In [ ]:
def send_reminder(message: str, *recipients: str, channel: str = "E-Mail") -> None:
    for recipient in recipients:
        print(f"{channel} an {recipient}: {message}")


send_reminder("Teammeeting 10 Uhr", "Ada", "Tim")
send_reminder("Teammeeting 10 Uhr", *team, channel="Teams")
send_reminder("Teammeeting 10 Uhr", team)  # Logikfehler: ein Empfänger, die ganze Liste
# channel ohne Namen ("Teams" an dritter Stelle) würde ebenfalls zum Empfänger.

## Funktionen sind Werte: Lambdas

Eine Funktion ist ein Objekt wie jedes andere (Notebook 02). Du kannst sie einer Variablen zuweisen oder einer anderen Funktion übergeben, **ohne** sie aufzurufen, also ohne Klammern:

In [ ]:
def shout(text: str) -> str:
    return text.upper() + "!"


action = shout  # ohne Klammern: die Funktion selbst, kein Aufruf
print(action("hallo"))
print(list(map(shout, ["ja", "nein"])))  # map ruft shout für jedes Element auf

Genau das nutzt `sorted(key=…)` aus Notebook 06: Du übergibst eine Funktion, und `sorted` ruft sie für jedes Element auf. Für solche kleinen Einmal-Funktionen gibt es die Kurzform **`lambda`**:

```
lambda parameter: ausdruck
```

Ein Lambda hat keinen Namen und besteht aus genau **einem Ausdruck**. Dessen Wert gibt es zurück, ganz ohne `return`. `max()` und `min()` verstehen `key=` genauso wie `sorted()`:

In [ ]:
people = [("Ada", 36), ("Linus", 55), ("Grace", 85)]

by_age = sorted(people, key=lambda person: person[1])
oldest = max(people, key=lambda person: person[1])
print(by_age)
print(oldest)

> Weise ein Lambda nicht einer Variablen zu (`square = lambda x: x * x`). Dafür ist `def` lesbarer, und die Funktion hat einen Namen, der im Traceback erscheint. Lambdas gehören direkt dorthin, wo eine kleine Funktion übergeben wird.

## Rekursion

Eine Funktion darf sich **selbst aufrufen**. Das heißt Rekursion. Sie passt zu Problemen, die sich in eine kleinere Version desselben Problems zerlegen lassen.

Beispiel Fakultät: 5! = 5 · 4 · 3 · 2 · 1, also 5! = 5 · 4!. Jede rekursive Funktion braucht zwei Teile:

1. **Abbruchbedingung** (Basisfall): Hier ist die Antwort direkt klar, 1! = 1.
2. **Rekursiver Aufruf** mit einem **kleineren** Problem: n! = n · (n − 1)!

In [ ]:
def factorial(n: int) -> int:
    if n <= 1:  # Abbruchbedingung
        return 1
    return n * factorial(n - 1)  # kleineres Problem


factorial(5)

So läuft `factorial(3)` ab. Jeder Aufruf wartet, bis der innere fertig ist:

```
factorial(3) = 3 * factorial(2)
                   factorial(2) = 2 * factorial(1)
                                      factorial(1) = 1
                   factorial(2) = 2 * 1 = 2
factorial(3) = 3 * 2 = 6
```

Fehlt die Abbruchbedingung, ruft sich die Funktion endlos auf. Python bricht nach etwa 1000 Ebenen mit einem `RecursionError` ab:

In [ ]:
def broken_factorial(n: int) -> int:
    return n * broken_factorial(n - 1)  # Abbruchbedingung vergessen


broken_factorial(5)

Richtig stark ist Rekursion bei **verschachtelten** Strukturen, deren Tiefe du vorher nicht kennst: Ordner mit Unterordnern, Menüs mit Untermenüs, Kommentare mit Antworten. Hier ein Menü als verschachtelte Liste. `isinstance(item, list)` prüft, ob ein Element selbst eine Liste ist:

In [ ]:
def count_items(nested: list) -> int:
    total = 0
    for item in nested:
        if isinstance(item, list):
            total += count_items(item)  # Unterliste: rekursiv zählen
        else:
            total += 1
    return total


menu = ["Start", ["Datei", ["Öffnen", "Speichern"]], ["Hilfe", ["Über"]]]
count_items(menu)

> Für einfache Wiederholungen wie die Fakultät ist eine Schleife in Python meist die bessere Wahl. Rekursion lohnt sich, wenn die Daten selbst verschachtelt sind.

## Generatoren mit `yield`

Eine Funktion mit `return` liefert **einen** Wert und ist fertig. Eine Funktion mit **`yield`** liefert Werte **nacheinander**: Sie gibt einen Wert ab, pausiert und macht beim nächsten Mal genau dort weiter. So eine Funktion heißt **Generator**.

In [ ]:
from collections.abc import Iterator


def countdown(start: int) -> Iterator[int]:
    while start > 0:
        yield start
        start -= 1
    print("Start!")


for number in countdown(3):
    print(number)

Der Rückgabetyp `Iterator[int]` sagt: Diese Funktion liefert nacheinander `int`-Werte. Denn ein Generator **ist** ein Iterator (Notebook 06): `for` holt sich mit `next()` Wert für Wert. Der Code im Generator läuft erst, wenn ein Wert angefordert wird:

In [ ]:
numbers = countdown(2)
print(numbers)        # ein Generator-Objekt, gelaufen ist noch nichts
print(next(numbers))
print(next(numbers))

Der große Vorteil: Ein Generator berechnet jeden Wert erst bei Bedarf und hält nicht alle gleichzeitig im Speicher. Er kann sogar **unendlich** viele Werte liefern; der Aufrufer entscheidet, wann Schluss ist:

In [ ]:
def even_numbers() -> Iterator[int]:
    number = 0
    while True:
        yield number
        number += 2


for number in even_numbers():
    if number > 10:
        break
    print(number, end=" ")

Für einfache Fälle gibt es den **Generator-Ausdruck**: Er sieht aus wie eine List Comprehension, aber mit runden Klammern. Er baut keine Liste, sondern liefert die Werte nacheinander. Ideal als Argument für `sum()`, `max()` oder `"".join()`; die Klammern des Funktionsaufrufs genügen dann:

In [ ]:
total = sum(n ** 2 for n in range(1_000_000))
total

## Gute Funktionen

- **Ein** Zweck pro Funktion, Name als Verb: `calculate_total`, `read_file`
- Lieber Werte zurückgeben als drucken
- Kurz halten: passt sie nicht auf einen Bildschirm, aufteilen
- Keine globalen Variablen verändern, alles über Parameter hereingeben

> Wie du Funktionen in eigene Dateien auslagerst und importierst, zeigt Notebook 08.

## Übungen

**Übung 3 (Rechnungszeilen mit Kopierfehler, leicht):** Den folgenden Code könnte eine KI geschrieben haben; er ist präpariert, ein Fehler ist eingebaut. Er gibt drei fast gleiche Rechnungszeilen aus. Finde den Fehler, lagere die Logik in `format_invoice_line(product, quantity, unit_price)` aus und gib zusätzlich eine Summenzeile aus.

In [ ]:
print(f"{'Kaffee':<10} {2:>3} x {4.50:>6.2f} € = {2 * 4.50:>7.2f} €")
print(f"{'Kuchen':<10} {1:>3} x {3.80:>6.2f} € = {2 * 3.80:>7.2f} €")
print(f"{'Wasser':<10} {3:>3} x {2.20:>6.2f} € = {3 * 2.20:>7.2f} €")

In [ ]:
# Fehler: Die Kuchen-Zeile rechnet 2 * 3.80, obwohl die Menge 1 ist.
def format_invoice_line(product: str, quantity: int, unit_price: float) -> str:
    total = quantity * unit_price
    return f"{product:<10} {quantity:>3} x {unit_price:>6.2f} € = {total:>7.2f} €"


items = [("Kaffee", 2, 4.50), ("Kuchen", 1, 3.80), ("Wasser", 3, 2.20)]
for item in items:
    print(format_invoice_line(*item))
grand_total = sum(quantity * price for _, quantity, price in items)
print(f"{'Summe':<27} {grand_total:>7.2f} €")

**Übung 4 (Passwortregeln als Funktion, mittel):** Der Startcode macht aus der Passwortprüfung aus 05 eine Funktion, die eine Liste der Mängel zurückgibt. Sage vorher, was die beiden Aufrufe ausgeben, führe den Code dann aus und erweitere ihn um eine Prüfung auf Groß- und Kleinbuchstaben, Type Hints und einen Docstring.

- **Untersuchen:** Was liefert `any(...)` für `"sommer"`? Warum wäre `for char in password: if char.isdigit(): return True else: return False` falsch?
- **Ändern:** Prüfe auf Groß- und Kleinbuchstaben und ergänze Type Hints und einen Docstring.

In [ ]:
def password_problems(password, min_length=12):
    problems = []
    if len(password) < min_length:
        problems.append(f"kürzer als {min_length} Zeichen")
    if not any(char.isdigit() for char in password):
        problems.append("Ziffer fehlt")
    return problems


print(password_problems("Sommer2026"))
print(password_problems("sommer", min_length=8))

In [ ]:
def password_problems(password: str, min_length: int = 12) -> list[str]:
    """Liefert die Mängel eines Passworts; leere Liste heißt sicher."""
    problems = []
    if len(password) < min_length:
        problems.append(f"kürzer als {min_length} Zeichen")
    if not any(char.isdigit() for char in password):
        problems.append("Ziffer fehlt")
    if not any(char.isupper() for char in password):
        problems.append("Großbuchstabe fehlt")
    if not any(char.islower() for char in password):
        problems.append("Kleinbuchstabe fehlt")
    return problems


print(password_problems("Sommer2026"))
print(password_problems("Sommer2026", min_length=8))

# Untersuchen: any(...) liefert für "sommer" False, kein Zeichen ist eine Ziffer.
# Die Schleife mit return True / else: return False entscheidet schon beim
# ersten Zeichen: "s" ist keine Ziffer, also False, obwohl später eine kommen könnte.

**Übung 5 (Stromkosten aller Geräte, mittel):** Der Startcode berechnet die Jahreskosten eines Kühlschranks. Sage die Ausgabe vorher, führe den Code dann aus und erweitere ihn: Ergänze Docstring und Type Hints und gib dann die Jahreskosten für Kühlschrank (150 W, 24 h), Fernseher (100 W, 4 h) und Router (10 W, 24 h) als Tabelle mit Summe aus.

- **Untersuchen:** Wie rufst du die Funktion mit einem Strompreis von 0,30 € auf? Was passiert, wenn sie `print` statt `return` benutzt?
- **Ändern:** Lege ein Geräte-Dictionary an, berechne die Kosten mit einer Comprehension und gib eine Tabelle mit Summe aus.

In [ ]:
def annual_cost(watts, hours_per_day, price_per_kwh=0.35):
    return watts / 1000 * hours_per_day * 365 * price_per_kwh


print(f"Kühlschrank: {annual_cost(150, 24):.2f} €")

In [ ]:
def annual_cost(
    watts: float, hours_per_day: float, price_per_kwh: float = 0.35
) -> float:
    """Jährliche Stromkosten eines Geräts in Euro."""
    return watts / 1000 * hours_per_day * 365 * price_per_kwh


# Untersuchen: anderer Strompreis per Name, annual_cost(150, 24, price_per_kwh=0.30).
# Mit print statt return liefert die Funktion None; eine Summe ginge dann nicht.
print(f"Kühlschrank bei 0,30 €: {annual_cost(150, 24, price_per_kwh=0.30):.2f} €")

devices = {"Kühlschrank": (150, 24), "Fernseher": (100, 4), "Router": (10, 24)}
costs = {name: annual_cost(watts, hours) for name, (watts, hours) in devices.items()}
for name, cost in costs.items():
    print(f"{name:<12} {cost:>7.2f} €")
print(f"{'Summe':<12} {sum(costs.values()):>7.2f} €")

**Übung 6 (Ordnergröße, schwer):** Ein Ordner ist ein Dictionary aus Namen und entweder Dateigröße in KB (`int`) oder Unterordner (`dict`). Der Startcode berechnet eine Ordnergröße. Sage vorher, was er für `home` liefert, führe den Code dann aus und finde heraus, was fehlt. Formuliere ihn rekursiv, sodass er alle Unterordner mitzählt.

- **Untersuchen:** Welche Dateien zählt der Startcode nicht? Was müsste für einen Unterordner passieren?
- **Ändern:** Rufe `folder_size` für Unterordner rekursiv auf.

In [ ]:
def folder_size(folder):
    total = 0
    for content in folder.values():
        if isinstance(content, int):
            total += content
    return total


home = {
    "cv.pdf": 120,
    "Fotos": {"urlaub.jpg": 3400, "2026": {"a.jpg": 2800}},
    "notes.txt": 4,
}
print(folder_size(home), "KB")

In [ ]:
def folder_size(folder: dict) -> int:
    total = 0
    for content in folder.values():
        if isinstance(content, dict):
            total += folder_size(content)  # Unterordner: rekursiv, Ergebnis addieren
        else:
            total += content
    return total


home = {
    "cv.pdf": 120,
    "Fotos": {"urlaub.jpg": 3400, "2026": {"a.jpg": 2800}},
    "notes.txt": 4,
}
print(folder_size(home), "KB")

# Untersuchen: Der Startcode zählt nur cv.pdf und notes.txt (124 KB).
# urlaub.jpg und a.jpg liegen in Unterordnern und fehlen. Für einen
# Unterordner muss die Funktion sich selbst aufrufen und das Ergebnis addieren.

## Zusammenfassung

- `def name(parameter):` definiert, `return` liefert das Ergebnis.
- Ohne `return` liefert eine Funktion `None`, ebenso Methoden wie `sort()`; `-> str | None` und `if x is None:` signalisieren „nichts gefunden“.
- Standardwerte `def f(x, rate=0.19)` und Aufrufe mit Namen `f(rate=0.07, x=1)`.
- `*args` sammelt beliebig viele Argumente in einem Tupel, `**kwargs` in einem Dictionary; `f(*liste)` packt beim Aufruf aus.
- Docstring und Type Hints machen Funktionen selbsterklärend.
- `pass` als Platzhalter für leere Blöcke.
- Funktionen sind Werte: übergeben ohne Klammern. `lambda x: ausdruck` für kleine Einmal-Funktionen, etwa als `key`.
- Rekursion: Abbruchbedingung plus Aufruf mit kleinerem Problem. Stark bei verschachtelten Daten.
- Generatoren liefern mit `yield` Werte nacheinander und erst bei Bedarf; `(x for x in …)` ist die Kurzform.